In [1]:
from datasets import load_from_disk
from gensim.corpora import Dictionary
import yaml

with open('./../../configs/model01_ast.yaml', 'r') as f:
  config = yaml.safe_load(f)

dataset = load_from_disk("../../data/processed/ast01")
code_dictionary = Dictionary.load('./../../data/processed/ast01/code_dictionary.pt')
docstring_dictionary = Dictionary.load('./../../data/processed/ast01/docstring_dictionary.pt')

c:\Users\Sean Andreini\Desktop\Unifi\Machine Learning for Software Analysis\code-summarization-mlsa-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import torch
import torch.nn as nn

class Encoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, dropout=0.2):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.embedding_dim = embedding_dim
        self.hidden = None # final hidden state
        self.cell = None
        self.basic_rnn = nn.LSTM(self.embedding_dim, self.hidden_dim, dropout=dropout, batch_first=True) # NLF

    def forward(self, X):
        embedded = self.embedding(X)
        batch_first_output, (self.hidden, self.cell) = self.basic_rnn(embedded) # NLH, 1NH, 1NH
        return batch_first_output, (self.hidden, self.cell)

In [3]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, bos_id, eos_id, pad_id, dropout = 0.2):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.embedding_dim = embedding_dim
        self.vocab_size = vocab_size
        self.hidden = None
        self.cell = None
        self.bos_id = bos_id
        self.eos_id = eos_id
        self.pad_id = pad_id
        self.basic_rnn = nn.LSTM(self.embedding_dim, self.hidden_dim, dropout=dropout, batch_first=True) # NLF
        self.output_layer = nn.Linear(self.hidden_dim, self.vocab_size)

    def init_hidden(self, encoder_states):
        self.hidden, self.cell = encoder_states

    def forward(self, X):
        # X is N, 1, F
        embedded = self.embedding(X)
        batch_first_output, (self.hidden, self.cell) = self.basic_rnn(embedded, (self.hidden, self.cell))
        logits = self.output_layer(batch_first_output)
        return logits, (self.hidden, self.cell)

In [4]:
class EncoderDecoder(nn.Module):
    def __init__(self, encoder, decoder, teacher_forcing_prob=0.5):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.teacher_forcing_prob = teacher_forcing_prob
        self.outputs = None

    def init_outputs(self, batch_size, target_len):
        device = next(self.parameters()).device
        # N, L, V (output is logits)
        self.outputs = torch.zeros(
            batch_size,
            target_len,
            self.decoder.vocab_size).to(device)

    def store_output(self, i, out):
        # Stores the output
        self.outputs[:, i:i+1, :] = out

    def forward(self, source_seq, target_seq):
      batch_size = source_seq.size(0)
      target_len = target_seq.size(1)

      _, (enc_hidden, enc_cell) = self.encoder(source_seq)

      self.decoder.init_hidden((enc_hidden, enc_cell))
      self.init_outputs(batch_size, target_len)

      dec_inputs = torch.full((batch_size, 1), 
                              self.decoder.bos_id,
                              dtype=torch.long).to(source_seq.device)
      
      for t in range(target_seq.size(1)):
        logits, _ = self.decoder(dec_inputs)
        self.outputs[:, t, :] = logits.squeeze(1)

        if self.training and torch.rand(1).item() < self.teacher_forcing_prob:
           dec_inputs = target_seq[:, t:t+1] # if teacher forcing, use actual next token as next input
        else:
           dec_inputs = logits.argmax(dim=-1) # else, use predicted token
        
      return self.outputs

In [5]:
from torch.nn.utils.rnn import pad_sequence
import torch

torch.cuda.empty_cache() # clears GPU memory
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [6]:
class CollateFn:
  def __init__(self, src_pad_id, tgt_pad_id):
    self.src_pad_id = src_pad_id
    self.tgt_pad_id = tgt_pad_id

  def __call__(self, batch):
    input_ids = [torch.tensor(x['input_ids'], dtype=torch.long) for x in batch]
    labels = [torch.tensor(x['labels'],    dtype=torch.long) for x in batch]

    input_ids = pad_sequence(
      input_ids,
      batch_first=True,
      padding_value=self.src_pad_id
    )

    labels = pad_sequence(
      labels,
      batch_first=True,
      padding_value=self.tgt_pad_id
    )

    return input_ids, labels
  
collate = CollateFn(
  src_pad_id=code_dictionary.token2id['[PAD]'],
  tgt_pad_id=docstring_dictionary.token2id['[PAD]']
)

In [13]:
import os

def save_checkpoint(epoch, model, optimizer, val_loss):
    os.makedirs(config['checkpoint_dir'], exist_ok=True)
    checkpoint_path = f'{config["checkpoint_dir"]}/best_model.pt'
    torch.save({'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': val_loss,
                }, checkpoint_path)
  
def load_checkpoint(model, optimizer):
    checkpoint_path = f'{config["checkpoint_dir"]}/best_model.pt'
    if(not os.path.exists(checkpoint_path)):
        print("No checkpoint found, starting from scratch")
        return 0
    
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch']
    print(f"Loaded checkpoint from epoch {start_epoch}")

    for state in optimizer.state.values():
      for k, v in state.items():
          if isinstance(v, torch.Tensor):
              state[k] = v.to(device)
    return start_epoch

In [8]:
from torch.utils.data import DataLoader
generator = torch.Generator()
generator.manual_seed(42)
torch.manual_seed(42)

train_dataset = dataset['train'].select(range(config['train_data_length']))
valid_dataset = dataset['valid'].select(range(config['valid_data_length']))

batch_size = config['batch_size']

train_dataloader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    pin_memory=True,
    collate_fn=collate,
    generator=generator
)

valid_dataloader = DataLoader(
    valid_dataset,
    batch_size=batch_size,
    shuffle=True,
    pin_memory=True,
    collate_fn=collate,
    generator=generator
)

code_vocab_size = len(code_dictionary.token2id)
docstring_vocab_size = len(docstring_dictionary.token2id)
embedding_dim = config['embedding_dim']
hidden_dim = config['hidden_dim']

encoder = Encoder(
  vocab_size=code_vocab_size,
  embedding_dim=embedding_dim,
  hidden_dim=hidden_dim
)

decoder = Decoder(
  vocab_size=docstring_vocab_size, 
  embedding_dim=embedding_dim, 
  hidden_dim=hidden_dim,
  bos_id=docstring_dictionary.token2id['[BOS]'],
  eos_id=docstring_dictionary.token2id['[EOS]'],
  pad_id=docstring_dictionary.token2id['[PAD]']
)

model = EncoderDecoder(
  encoder=encoder,
  decoder=decoder,
  teacher_forcing_prob=config['teacher_forcing_prob']
)



loss = nn.CrossEntropyLoss(ignore_index=docstring_dictionary.token2id['[PAD]'])
optimizer = torch.optim.Adam(model.parameters(), lr=config['learning_rate'])

c:\Users\Sean Andreini\Desktop\Unifi\Machine Learning for Software Analysis\code-summarization-mlsa-project\.venv\Lib\site-packages\torch\nn\modules\rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn(


In [ ]:
epochs = config['num_epochs']
start_epoch = load_checkpoint(model, optimizer)+1


#device = 'cpu'
model.to(device)
best_loss = float('inf')
counter = 0

for epoch in range(start_epoch, epochs):
  model.train()
  batch_losses = []
  for input, labels in train_dataloader:
    input = input.to(device, non_blocking=True)
    labels = labels.to(device, non_blocking=True)
    optimizer.zero_grad()
    y_pred = model(input, labels)
    y_pred = y_pred.permute(0, 2, 1)  # N, V, L
    single_loss = loss(y_pred, labels)
    single_loss.backward()
    optimizer.step()
    batch_losses.append(single_loss.item())
  
  if(epoch % 5 == 0):
    print(f"Epoch {epoch:3}, Training Loss: {sum(batch_losses)/len(batch_losses):10.8f}")

  model.eval()
  with torch.no_grad():
    val_losses = []
    for input, labels in valid_dataloader:
      input = input.to('cuda' if torch.cuda.is_available() else 'cpu', non_blocking=True)
      labels = labels.to('cuda' if torch.cuda.is_available() else 'cpu', non_blocking=True)
      y_pred = model(input, labels)
      y_pred = y_pred.permute(0, 2, 1)  # N, V, L
      single_loss = loss(y_pred, labels)
      val_losses.append(single_loss.item())

  if epoch % 5 == 0:
    print(f"Epoch {epoch:3}, Validation Loss: {sum(val_losses)/len(val_losses):10.8f}")

  if(len(val_losses) > 0 and sum(val_losses)/len(val_losses) < best_loss):
    best_loss = sum(val_losses)/len(val_losses)
    counter = 0
    save_checkpoint(epoch, model, optimizer, best_loss)
    
  else:
    counter += 1
    if counter >= config['patience']:
      print("Early stopping")
      break
  
  torch.cuda.empty_cache() # clears GPU memory
  del single_loss
  del y_pred
  del input
  del labels

Loaded checkpoint from epoch 1
Epoch   5, Training Loss: 4.10286917
Epoch   5, Validation Loss: 6.86825232
Epoch  10, Training Loss: 2.26074925
Epoch  10, Validation Loss: 7.70185417
Epoch  15, Training Loss: 1.41115792
Epoch  15, Validation Loss: 8.61783052
Epoch  20, Training Loss: 1.05832755
Epoch  20, Validation Loss: 9.39093637
Early stopping


In [24]:
import ast

# Function to convert code string to AST
def code_to_ast(code_string):
  try:
    return ast.parse(code_string)
  except SyntaxError:
    return None # some code snippets are not valid (python 2 instead of python 3)

In [22]:
### Linearizing the AST (by passing first node) into a list of tokens
def linearize_ast(node, tokens):
  if node is None:
    return

  tokens.append(type(node).__name__)

  # specific node types
  # if it's a name we add the variable name
  if isinstance(node, ast.Name):
    tokens.append(f"VAR_{node.id}")

  # if it's a constant we add a placeholder to not write actual values
  elif isinstance(node, ast.Constant):
    tokens.append("CONST")

  # if it's a function argument we add the argument name
  elif isinstance(node, ast.arg):
    tokens.append(f"ARG_{node.arg}")

  for child in ast.iter_child_nodes(node):
    linearize_ast(child, tokens)

In [43]:
load_checkpoint(model, optimizer)
model.to(device)

model.eval()

linearized_tree = []
linearize_ast(code_to_ast("def add(a, b): return a+b"), linearized_tree)
print(linearized_tree)

Loaded checkpoint from epoch 2
['Module', 'FunctionDef', 'arguments', 'arg', 'ARG_a', 'arg', 'ARG_b', 'Return', 'BinOp', 'Name', 'VAR_a', 'Load', 'Add', 'Name', 'VAR_b', 'Load']


In [44]:
def predict_docstring(model, code_dictionary, docstring_dictionary, max_len=50):
    device = next(model.parameters()).device
    model.eval()
    
    input_ids = [code_dictionary.token2id.get(token, code_dictionary.token2id['[UNK]'])
                 for token in linearized_tree]
    print(input_ids)
    input_tensor = torch.tensor([input_ids], dtype=torch.long, device=device)
    
    encoder_outputs, (enc_hidden, enc_cell) = model.encoder(input_tensor)
    model.decoder.init_hidden((enc_hidden, enc_cell))
    
    dec_input = torch.full((1, 1), model.decoder.bos_id, dtype=torch.long, device=device)
    generated_ids = []
    
    for _ in range(max_len):
        logits, (enc_hidden, enc_cell) = model.decoder(dec_input)
        next_token_id = logits.argmax(dim=-1)
        if next_token_id.item() == model.decoder.eos_id:
            break
        generated_ids.append(next_token_id.item())
        dec_input = next_token_id
    
    return generated_ids

In [45]:
ids = predict_docstring(model, code_dictionary, docstring_dictionary)
string = " ".join([docstring_dictionary.get(idx, '[UNK]') for idx in ids])

print(string)

[12, 10, 21, 20, 579, 20, 580, 14, 5, 13, 583, 11, 444630, 13, 584, 11]
This a the to the . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .


Now, we can see that the model does not behave that well. As we anticipated a couple of notebooks ago, we can definitely improve this encoder-decoder with some attention.

In [14]:
import numpy as np
import torch.nn.functional as F

In [15]:
class Attention(nn.Module):
  def __init__(self, hidden_dim, input_dim=None, proj_values=False):
    super().__init__()
    self.d_k = hidden_dim
    self.input_dim = hidden_dim if input_dim is None else input_dim
    self.proj_values = proj_values
    # Affine transformations for Q, K, and V
    self.linear_query = nn.Linear(self.input_dim, hidden_dim)
    self.linear_key = nn.Linear(self.input_dim, hidden_dim)
    self.linear_value = nn.Linear(self.input_dim, hidden_dim)
    self.alphas = None

  def init_keys(self, keys):
    self.keys = keys
    self.proj_keys = self.linear_key(self.keys)
    self.values = self.linear_value(self.keys) \
                  if self.proj_values else self.keys

  def score_function(self, query):
    proj_query = self.linear_query(query)
    # scaled dot product
    # N, 1, H x N, H, L -> N, 1, L
    dot_products = torch.bmm(proj_query, self.proj_keys.permute(0, 2, 1))
    scores =  dot_products / np.sqrt(self.d_k)
    return scores

  def forward(self, query, mask=None):
    # Query is batch-first N, 1, H
    scores = self.score_function(query) # N, 1, L
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    alphas = F.softmax(scores, dim=-1) # N, 1, L
    self.alphas = alphas.detach()

    # N, 1, L x N, L, H -> N, 1, H
    context = torch.bmm(alphas, self.values)
    return context

In [16]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, bos_id, eos_id, pad_id, dropout = 0.2):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.embedding_dim = embedding_dim
        self.vocab_size = vocab_size
        self.hidden = None
        self.cell = None
        self.bos_id = bos_id
        self.eos_id = eos_id
        self.pad_id = pad_id
        self.attention = Attention(hidden_dim)
        self.basic_rnn = nn.LSTM(self.embedding_dim, self.hidden_dim, dropout=dropout, batch_first=True) # NLF
        self.output_layer = nn.Linear(2*self.hidden_dim, self.vocab_size) # 2* because of context+query

    def init_hidden(self, encoder_states):
        self.hidden, self.cell = encoder_states
        self.attention.init_keys(encoder_states[0].permute(1,0,2)) # attention wants batch first

    def forward(self, X, mask=None):
        # X is N, 1, F
        embedded = self.embedding(X)
        batch_first_output, (self.hidden, self.cell) = self.basic_rnn(embedded, (self.hidden, self.cell))
        
        # attention 
        query = batch_first_output[:, -1:, :]
        context = self.attention(query, mask=mask)
        concatenated = torch.cat([context, query], axis=-1)
        logits = self.output_layer(concatenated)
        return logits, (self.hidden, self.cell)

In [17]:
class EncoderDecoderAttn(EncoderDecoder):
    def __init__(self, encoder, decoder, teacher_forcing_prob=0.5):
        super().__init__(encoder, decoder, teacher_forcing_prob)
        self.alphas = None

    def init_outputs(self, batch_size, target_len):
        device = next(self.parameters()).device
        # N, L, V (output is logits)
        self.outputs = torch.zeros(
            batch_size,
            target_len,
            self.decoder.vocab_size).to(device)
        # N, L (target), L (source)
        self.alphas = torch.zeros(batch_size,
            target_len,
            self.input_len).to(device)

    def store_output(self, i, out):
        # Stores the output
        self.outputs[:, i:i+1, :] = out
        self.alphas[:, i:i+1, :] = self.decoder.attn.alphas

        
    def forward(self, source_seq, target_seq, source_mask=None):
      batch_size = source_seq.size(0)
      target_len = target_seq.size(1)
      self.input_len = source_seq.size(1)

      encoder_outputs, (enc_hidden, enc_cell) = self.encoder(source_seq)

      self.decoder.init_hidden((enc_hidden, enc_cell))
      self.decoder.attention.init_keys(encoder_outputs)
      self.init_outputs(batch_size, target_len)

      dec_inputs = torch.full((batch_size, 1), 
                              self.decoder.bos_id,
                              dtype=torch.long).to(source_seq.device)
      
      for t in range(target_seq.size(1)):
        logits, _ = self.decoder(dec_inputs, mask=source_mask)
        self.outputs[:, t, :] = logits.squeeze(1)
        self.alphas[:, t:t+1, :] = self.decoder.attention.alphas

        if self.training and torch.rand(1).item() < self.teacher_forcing_prob:
           dec_inputs = target_seq[:, t:t+1] # if teacher forcing, use actual next token as next input
        else:
           dec_inputs = logits.argmax(dim=-1) # else, use predicted token
        
      return self.outputs

In [18]:
from torch.utils.data import DataLoader
generator = torch.Generator()
generator.manual_seed(42)
torch.manual_seed(42)

train_dataset = dataset['train'].select(range(config['train_data_length']))
valid_dataset = dataset['valid'].select(range(config['valid_data_length']))

batch_size = config['batch_size']

train_dataloader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    pin_memory=True,
    collate_fn=collate,
    generator=generator
)

valid_dataloader = DataLoader(
    valid_dataset,
    batch_size=batch_size,
    shuffle=True,
    pin_memory=True,
    collate_fn=collate,
    generator=generator
)

code_vocab_size = len(code_dictionary.token2id)
docstring_vocab_size = len(docstring_dictionary.token2id)
embedding_dim = config['embedding_dim']
hidden_dim = config['hidden_dim']

encoder = Encoder(
  vocab_size=code_vocab_size,
  embedding_dim=embedding_dim,
  hidden_dim=hidden_dim
)

decoder = Decoder(
  vocab_size=docstring_vocab_size, 
  embedding_dim=embedding_dim, 
  hidden_dim=hidden_dim,
  bos_id=docstring_dictionary.token2id['[BOS]'],
  eos_id=docstring_dictionary.token2id['[EOS]'],
  pad_id=docstring_dictionary.token2id['[PAD]']
)

model = EncoderDecoderAttn(
  encoder=encoder,
  decoder=decoder,
  teacher_forcing_prob=config['teacher_forcing_prob']
)



loss = nn.CrossEntropyLoss(ignore_index=docstring_dictionary.token2id['[PAD]'])
optimizer = torch.optim.Adam(model.parameters(), lr=config['learning_rate'])

c:\Users\Sean Andreini\Desktop\Unifi\Machine Learning for Software Analysis\code-summarization-mlsa-project\.venv\Lib\site-packages\torch\nn\modules\rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn(


In [20]:
config['checkpoint_dir'] = './../../checkpoints/model01/ast_attention'
epochs = config['num_epochs']
start_epoch = load_checkpoint(model, optimizer)+1

#device = 'cpu'
model.to(device)
best_loss = float('inf')
counter = 0

for epoch in range(start_epoch, epochs):
  model.train()
  batch_losses = []
  for input, labels in train_dataloader:
    input = input.to(device, non_blocking=True)
    labels = labels.to(device, non_blocking=True)
    optimizer.zero_grad()
    y_pred = model(input, labels)
    y_pred = y_pred.permute(0, 2, 1)  # N, V, L
    single_loss = loss(y_pred, labels)
    single_loss.backward()
    optimizer.step()
    batch_losses.append(single_loss.item())
  
  if(epoch % 5 == 0):
    print(f"Epoch {epoch:3}, Training Loss: {sum(batch_losses)/len(batch_losses):10.8f}")

  model.eval()
  with torch.no_grad():
    val_losses = []
    for input, labels in valid_dataloader:
      input = input.to('cuda' if torch.cuda.is_available() else 'cpu', non_blocking=True)
      labels = labels.to('cuda' if torch.cuda.is_available() else 'cpu', non_blocking=True)
      y_pred = model(input, labels)
      y_pred = y_pred.permute(0, 2, 1)  # N, V, L
      single_loss = loss(y_pred, labels)
      val_losses.append(single_loss.item())

  if epoch % 5 == 0:
    print(f"Epoch {epoch:3}, Validation Loss: {sum(val_losses)/len(val_losses):10.8f}")

  if(len(val_losses) > 0 and sum(val_losses)/len(val_losses) < best_loss):
    best_loss = sum(val_losses)/len(val_losses)
    counter = 0
    save_checkpoint(epoch, model, optimizer, best_loss)
    print(f"Saved new best model (epoch {epoch})")
    
  else:
    counter += 1
    if counter >= config['patience']:
      print("Early stopping")
      break
  
  torch.cuda.empty_cache() # clears GPU memory
  del single_loss
  del y_pred
  del input
  del labels

No checkpoint found, starting from scratch
Saved new best model (epoch 1)
Epoch   5, Training Loss: 4.36810608
Epoch   5, Validation Loss: 7.73747898
Early stopping


In [44]:
load_checkpoint(model, optimizer)
model.to(device)

model.eval()

linearized_tree = []
linearize_ast(code_to_ast("def print(): print('ciao')"), linearized_tree)
print(linearized_tree)

Loaded checkpoint from epoch 1
['Module', 'FunctionDef', 'arguments', 'Expr', 'Call', 'Name', 'VAR_print', 'Load', 'Constant', 'CONST']


In [45]:
def predict_docstring(model, code_dictionary, docstring_dictionary, max_len=50):
    device = next(model.parameters()).device
    model.eval()
    
    input_ids = [code_dictionary.token2id.get(token, code_dictionary.token2id['[UNK]'])
                 for token in linearized_tree]
    input_tensor = torch.tensor([input_ids], dtype=torch.long, device=device)
    
    encoder_outputs, (enc_hidden, enc_cell) = model.encoder(input_tensor)
    model.decoder.init_hidden((enc_hidden, enc_cell))
    model.decoder.attention.init_keys(encoder_outputs)
    
    dec_input = torch.full((1, 1), model.decoder.bos_id, dtype=torch.long, device=device)
    generated_ids = []
    
    for _ in range(max_len):
        
        logits, (enc_hidden, enc_cell) = model.decoder(dec_input)
        next_token_id = logits.argmax(dim=-1)
        if next_token_id.item() == model.decoder.eos_id:
            break
        generated_ids.append(next_token_id.item())
        dec_input = next_token_id
        alphas = model.decoder.attention.alphas  # shape: [1, 1, seq_len]
        print(docstring_dictionary.get(next_token_id.item(), '[UNK]'))
        print(alphas)
    
    return generated_ids

In [46]:
ids = predict_docstring(model, code_dictionary, docstring_dictionary)
string = " ".join([docstring_dictionary.get(idx, '[UNK]') for idx in ids])

print(string)

Returns
tensor([[[0.0265, 0.1633, 0.0402, 0.0889, 0.0185, 0.0650, 0.0007, 0.3636,
          0.0184, 0.2149]]], device='cuda:0')
a
tensor([[[0.0310, 0.1126, 0.0424, 0.1009, 0.0381, 0.0906, 0.0034, 0.3649,
          0.0358, 0.1804]]], device='cuda:0')
list
tensor([[[0.0371, 0.1387, 0.0492, 0.0979, 0.0368, 0.0855, 0.0040, 0.2631,
          0.0287, 0.2589]]], device='cuda:0')
of
tensor([[[9.5603e-03, 1.5004e-01, 1.6732e-02, 5.6796e-02, 5.1083e-03,
          3.0012e-02, 3.6855e-05, 3.9678e-01, 4.5669e-03, 3.3036e-01]]],
       device='cuda:0')
the
tensor([[[6.1669e-03, 1.3719e-01, 1.1672e-02, 4.9504e-02, 3.1876e-03,
          2.3448e-02, 1.8725e-05, 3.9694e-01, 3.1293e-03, 3.6874e-01]]],
       device='cuda:0')
.
tensor([[[4.8997e-03, 1.4401e-01, 8.8104e-03, 4.4329e-02, 2.2123e-03,
          2.1383e-02, 6.1178e-06, 5.0925e-01, 2.3639e-03, 2.6274e-01]]],
       device='cuda:0')
.
tensor([[[8.4582e-03, 1.3013e-01, 1.5141e-02, 6.1543e-02, 5.8688e-03,
          3.5927e-02, 5.2863e-05, 4.7427e-0